# 07 Ablations (degradation off)

Which parts of the DPC design matter? Three variants were trained exactly like the selected candidate (horizon 144, 30-70%
dead zone, 1000 steps, seeds 0, 1 and 2), each with one part removed. The protocol was written before any ablation run.

| Variant | Soft SoC and voltage penalties in the loss | Safety layer (energy limit, P/Q box, S scaling) |
|---|---|---|
| selected (reference) | yes | yes |
| projection-only | **no** | yes |
| penalty-only | yes, **plus a soft inverter-limit penalty (decision D1)** | **no** |
| no-constraints | **no** | **no** |

Degradation and C-rate terms are off in every variant. The policy's output layer is linear, so without the safety layer
nothing bounds P, Q or SoC; nothing is clipped. **Steps that break an inverter or SoC limit are physically impossible
operation**; they are reported as shares, and the objective of such a variant is not achievable performance. The
comparison objective is the same for all variants (eq. (16) terms only), so penalty terms are counted even for variants
that did not train on them, and the D1 term is not part of it.

D1 is a change from the original code: without it, "penalty-only" would not penalise the limits that the safety layer
enforces, and the ablation would mostly measure unlimited reactive power.

```{include} ../_caveats.md
```

In [ ]:
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from common import *
ab = load("m3_ablations.json")
display(Markdown(provenance(ab, "m3_ablations.json")))
rows = []
for name, t in ab["training"].items():
    r = t["records"][-1]
    rows.append({"run": name, "unstable": fmt(r["unstable"]), "max grad norm": fmt(r["max_grad_norm"], 1),
                 "truncated rerun": "no" if len(t["records"]) == 1 else "yes", "minutes": fmt(r["seconds"] / 60, 1)})
display(Markdown("**Training stability** (same pre-specified rule as Experiment 2)"))
table(rows, index="run")

## Over seeds: median [min-max]

In [ ]:
variants = {"selected (penalties + safety layer)": ab["selected_candidate_reference"]["over_seeds"],
            "projection-only": ab["variants"]["projection-only"]["over_seeds"],
            "penalty-only (with D1)": ab["variants"]["penalty-only-no-deg"]["over_seeds"],
            "no-constraints": ab["variants"]["no-constraints-no-deg"]["over_seeds"]}
metrics = [("objective", "objective", 5, False), ("vvr_share_of_steps", "VVR", 3, True),
           ("share_battery_steps_breaking_inverter_limits", "steps breaking an inverter limit", 1, True),
           ("share_battery_steps_outside_soc_limits", "steps outside SoC 10-90%", 2, True),
           ("mean_network_loss_kw", "losses (kW)", 1, False), ("mean_abs_p_mw", "mean |P| (MW)", 4, False),
           ("mean_abs_q_mvar", "mean |Q| (Mvar)", 3, False), ("soc_min", "SoC min", 2, False), ("soc_max", "SoC max", 2, False)]
def mr(m, digits, as_pct):
    if m is None:
        return "—"
    f = (lambda y: pct(y, digits)) if as_pct else (lambda y: fmt(y, digits))
    return f"{f(m['median'])} [{f(m['min'])}-{f(m['max'])}]"
rows = []
for reg, rl in REGIMES.items():
    for name, o in variants.items():
        row = {"regime": rl, "variant": name}
        for key, label, digits, as_pct in metrics:
            row[label] = mr(o[reg].get(key), digits, as_pct)
        rows.append(row)
table(rows).set_index(["regime", "variant"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
names = list(variants)
for ax, reg in zip(axes, REGIMES):
    for y, name in enumerate(names):
        m = variants[name][reg]["share_battery_steps_breaking_inverter_limits"]
        if m is None:
            continue
        ax.barh(y, 100 * m["median"], color=SERIES[0], height=0.6)
        ax.plot([100 * m["min"], 100 * m["max"]], [y, y], color=INK, linewidth=1.5)
        ax.text(100 * m["max"], y, f"  {100 * m['median']:.1f}%", va="center", fontsize=8)
    ax.set_yticks(range(len(names)), names); ax.invert_yaxis(); ax.set_xlim(0, 100)
    ax.set_xlabel("battery steps breaking an inverter limit (%)\nbar: median, line: min-max over seeds")
    ax.set_title(REGIMES[reg], loc="left")
fig.tight_layout(); plt.show()

## Every seed

In [ ]:
labels = {"projection-only": "projection-only", "penalty-only-no-deg": "penalty-only (with D1)", "no-constraints-no-deg": "no-constraints"}
rows = []
for v, label in labels.items():
    for seed, c in ab["variants"][v]["per_seed"].items():
        for reg, rl in REGIMES.items():
            r = c["regimes"][reg]; s = r["summary"]
            rows.append({"variant": label, "seed": seed.replace("seed_", ""), "regime": rl, "objective": fmt(r["objective"]),
                         "mean |P| per battery (MW)": ", ".join(fmt(x, 4) for x in r["mean_abs_p_mw"]),
                         "mean |Q| per battery (Mvar)": ", ".join(fmt(x, 3) for x in r["mean_abs_q_mvar"]),
                         "inverter-limit breaks": pct(s["share_of_battery_steps_breaking_inverter_limits"], 2),
                         "SoC outside limits": pct(s["share_of_battery_steps_outside_soc_limits"], 2),
                         "VVR": pct(s["vvr_share_of_steps_with_any_violation"], 3),
                         "end-of-day SoC": f"{fmt(r['end_of_day_soc_min'], 2)} - {fmt(r['end_of_day_soc_max'], 2)}"})
table(rows).set_index(["variant", "seed", "regime"])

**Reading (descriptive only):** with the safety layer, removing the soft penalties changes the objective very little,
because on validation the voltages and SoC stay away from their limits. Without the safety layer, the D1 penalty keeps
limit violations small but not zero, and the variant without any constraint has the lowest objective only because it
exceeds the inverter rating on most steps, which a real inverter cannot do. Validation part, one network; not test results.

In [ ]:
import json
outside = [o[reg]["share_battery_steps_outside_soc_limits"] for o in variants.values() for reg in REGIMES]
if all(m is not None and m["max"] == 0 for m in outside):
    bat = json.loads((ROOT / "configs" / "m3.json").read_text())["battery"]
    display(Markdown(f"No variant left the SoC range {bat['soc_min']:.0%}-{bat['soc_max']:.0%} (0.00% of battery steps in all variants, seeds and regimes), so these "
                     "runs do not test the energy-limit part of the safety layer; they test its inverter-limit part."))
display(Markdown("Scope: the ablations cover constraint handling only (the safety layer and the soft penalties). The "
                 "differentiable solver, the horizon, the terminal term, the forecast (none) and degradation (off) are not ablated."))